# Opportunity audit for the adjoint allocator (run after the production pilot)

**Why this exists.** Run `droid100_adjoint_20260929T070629Z` passed the dynamics gate but failed the allocation gate: on the test split both learned allocators had higher regret than a single random draw, top-1 accuracy was below chance, and neither allocator ever chose candidates 1 or 3 (which the oracle picked on 65% of test windows). See `docs/research-notes/2026-09-29-droid100-pilot-findings.md`.

The pilot only logs the chosen candidate per window, so these questions can't be answered from what's already saved:

1. How much can *any* per-window policy win over the best **constant** candidate (research plan §7.2B opportunity gate)?
2. Does the **exact** co-state, used in the same first-order score `-<λ, δ_k> - c_k`, beat random? If not, the linearisation is the problem, not the amortised estimator.
3. Is the train split's gain distribution different from the test split's (which candidate wins, how large the spread is)?
4. Are the candidate effects distinct directions, or near-duplicates / sign-flipped pairs?

**How to run.** Open the production pilot, run it through section 5 (allocator training) so `model`, `teacher`, `exact_adjoint_targets`, the three datasets, `CFG`, `DEVICE`, `SELECTED_BATCH_SIZE` and `RUN_DIR` exist in the kernel, then paste/run these cells in the same runtime. Nothing here trains or changes weights. Outputs go to `RUN_DIR/diagnostics/`.

In [ ]:
# Make the repo's analysis helpers importable (pure numpy/pandas).
import os, subprocess, sys
REPO_DIR = '/content/para_001'
if not os.path.exists(REPO_DIR):
    # Private repo: run `!gh auth login` first, or upload src/adjointrwm to /content/para_001/src.
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/romusking/para_001', REPO_DIR], check=False)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
from adjointrwm.analysis import (
    opportunity_audit, policy_regret, first_order_scores, compare_distributions,
)
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

DIAG_DIR = RUN_DIR / 'diagnostics'
DIAG_DIR.mkdir(parents=True, exist_ok=True)
print('Writing diagnostics to', DIAG_DIR)

In [ ]:
@torch.no_grad()
def _predict_scores(latent, effects):
    batch = latent.shape[0]
    budget_fraction = torch.full((batch,), 1.0 / model.num_candidates, device=DEVICE)
    horizon_fraction = torch.ones(batch, device=DEVICE)
    predicted_costate = model.predict_costate(latent, budget_fraction, horizon_fraction)
    critic_scores = model.predict_critic(latent, effects)
    return predicted_costate, critic_scores


def collect_allocation_tensors(dataset, max_windows=None):
    """Full per-window gain matrix plus everything needed to re-score offline."""
    model.eval()
    loader = DataLoader(dataset, batch_size=SELECTED_BATCH_SIZE, shuffle=False, num_workers=0)
    keep = {k: [] for k in ('exact_gain', 'exact_costate', 'predicted_costate',
                            'critic_scores', 'effects', 'stop_objective')}
    episode_ids, window_starts, seen = [], [], 0
    for batch in loader:
        with torch.enable_grad():
            targets = exact_adjoint_targets(batch)
        predicted_costate, critic_scores = _predict_scores(targets['latent'], targets['effects'])
        keep['exact_gain'].append(targets['exact_gain'].float().cpu())
        keep['exact_costate'].append(targets['exact_costate'].float().cpu())
        keep['predicted_costate'].append(predicted_costate.float().cpu())
        keep['critic_scores'].append(critic_scores.float().cpu())
        keep['effects'].append(targets['effects'].float().cpu())
        keep['stop_objective'].append(targets['stop_objective'].float().cpu())
        episode_ids += list(batch['episode_id'])
        window_starts += [int(x) for x in batch['window_start']]
        seen += len(batch['episode_id'])
        if max_windows and seen >= max_windows:
            break
    out = {k: torch.cat(v).numpy() for k, v in keep.items()}
    out['episode_id'] = np.array(episode_ids)
    out['window_start'] = np.array(window_starts)
    return out


SPLITS = {'train': train_dataset, 'validation': val_dataset, 'test': test_dataset}
# The train split is ~5.9k windows; the exact adjoint costs ~15 ms per batch, so this is minutes on an L4.
tensors = {name: collect_allocation_tensors(ds) for name, ds in SPLITS.items()}
for name, t in tensors.items():
    np.savez_compressed(DIAG_DIR / f'allocation_tensors_{name}.npz', **t)
    print(name, t['exact_gain'].shape)

In [ ]:
weighted_costs = (np.asarray(CFG.candidate_costs) * CFG.candidate_cost_weight)

def evaluate_split(t):
    gain = t['exact_gain']
    n, k = gain.shape
    report = opportunity_audit(gain, candidate_costs=weighted_costs).to_dict()

    exact_first_order = first_order_scores(t['exact_costate'], t['effects'], weighted_costs)
    amortised_first_order = first_order_scores(t['predicted_costate'], t['effects'], weighted_costs)
    policies = {
        'oracle': gain.argmax(1),
        'exact_costate_first_order': exact_first_order.argmax(1),
        'amortised_costate': amortised_first_order.argmax(1),
        'critic': t['critic_scores'].argmax(1),
        **{f'always_c{j}': np.full(n, j) for j in range(k)},
    }
    rows = []
    for name, choice in policies.items():
        rows.append({
            'policy': name,
            'mean_regret': float(policy_regret(gain, choice).mean()),
            'top1': float((choice == gain.argmax(1)).mean()),
            'choice_share': np.bincount(choice, minlength=k).tolist(),
        })
    # Expected regret of uniform random choice (not a single draw).
    rows.append({'policy': 'random_expected',
                 'mean_regret': float((gain.max(1, keepdims=True) - gain).mean()),
                 'top1': 1.0 / k, 'choice_share': None})
    report['policies'] = rows

    # How good is the first-order (linearised) predictor of the true gain?
    stop = t['stop_objective'][:, None]
    linear_pred = -np.einsum('nd,nkd->nk', t['exact_costate'], t['effects'])
    true_improvement = gain + weighted_costs[None, :]
    report['first_order_vs_exact_corr'] = float(np.corrcoef(linear_pred.ravel(), true_improvement.ravel())[0, 1])

    # Are the candidate effects distinct directions?
    eff = t['effects'] / (np.linalg.norm(t['effects'], axis=-1, keepdims=True) + 1e-12)
    report['mean_effect_cosine'] = (np.einsum('nid,njd->ij', eff, eff) / n).round(3).tolist()
    report['mean_effect_norm'] = np.linalg.norm(t['effects'], axis=-1).mean(0).tolist()
    return report

reports = {name: evaluate_split(t) for name, t in tensors.items()}
reports['train_vs_test'] = compare_distributions(tensors['train']['exact_gain'], tensors['test']['exact_gain'])
atomic_write_json(DIAG_DIR / 'opportunity_audit.json', reports)

for name in SPLITS:
    r = reports[name]
    print(f'\n=== {name} ===  opportunity gate passes: {r["passes"]}  ({r["reason"]})')
    print(pd.DataFrame(r['policies']).to_string(index=False))
    print('first-order vs exact correlation:', round(r['first_order_vs_exact_corr'], 3))
    print('oracle share:', np.round(r['oracle_share'], 3), ' median spread:', round(r['median_gain_spread'], 4),
          ' cost gap:', r['cost_gap'])
print('\nTrain vs test:', json.dumps(reports['train_vs_test'], indent=1))

## Reading the result

| Observation | What it means | Next action |
|---|---|---|
| `headroom_over_best_fixed` ≈ 0 on test | No per-window choice to learn; benchmark is non-diagnostic (plan §7.2B) | Redesign candidates (larger/more distinct effects, real sensing ports) before any H2 claim |
| `always_cX` beats both learned allocators | Allocators learned a train-split preference that doesn't transfer | Check `train_vs_test`; rebalance or add split-robust features |
| `exact_costate_first_order` ≫ `amortised_costate` | Mechanism is sound, estimator is weak | Improve co-state estimator (capacity, direction+magnitude weighting) |
| `exact_costate_first_order` ≈ random | First-order linearisation doesn't rank candidates here | Candidate effects too large/nonlinear for a linear score, or objective gradient uninformative; test smaller δ or second-order terms |
| Effect cosines near ±1 off-diagonal | Candidates are duplicates / sign-flipped pairs | Enforce diversity or use structurally different candidates |

Commit `opportunity_audit.json` (small) to `results/runs/<run_id>/diagnostics/`. Keep the `.npz` tensors on Drive and list their SHA-256 in the run README.